In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/omarcodeplay/gm-dataset/config.json
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/bench.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/build_data.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/download_data.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/encode.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/engine.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/gauntlet.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/model.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/selftest.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/sign_check.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/sync.py
/kaggle/input/datasets/omarcodeplay/gm-dataset/src/train.py


In [3]:
import os, shutil, pathlib

SRC_IN  = "/kaggle/input/datasets/omarcodeplay/gm-dataset/src"
SRC_OUT = "/kaggle/working/src"

# 1. Verify input mount exists
if not os.path.isdir(SRC_IN):
    # Fallback: search common dataset paths
    import glob
    candidates = glob.glob("/kaggle/input/**/src", recursive=True)
    if candidates:
        SRC_IN = candidates[0]
        print(f"found src at: {SRC_IN}")
    else:
        raise SystemExit(
            f"Source not found at {SRC_IN}.\n"
            f"Check 'Add Data' in the Kaggle notebook sidebar."
        )

# 2. Copy everything
os.makedirs(SRC_OUT, exist_ok=True)
copied = []
for f in pathlib.Path(SRC_IN).glob("*.py"):
    dst = pathlib.Path(SRC_OUT) / f.name
    shutil.copy(f, dst)
    copied.append(f.name)

# 3. Also copy config.json if present (train.py looks for it via sync)
cfg = pathlib.Path(SRC_IN).parent / "config.json"
if cfg.exists():
    shutil.copy(cfg, "/kaggle/working/config.json")
    print(f"copied config.json")

print(f"✅ Copied {len(copied)} files to {SRC_OUT}:")
for f in sorted(copied):
    print(f"  {f}")

copied config.json
✅ Copied 11 files to /kaggle/working/src:
  bench.py
  build_data.py
  download_data.py
  encode.py
  engine.py
  gauntlet.py
  model.py
  selftest.py
  sign_check.py
  sync.py
  train.py


In [6]:
%%bash
pip install -q --upgrade python-chess huggingface_hub pandas numpy zstandard orjson

In [16]:
import os, importlib, sys

# 1. Force reload sync so it picks up the CURRENT env
if "sync" in sys.modules:
    del sys.modules["sync"]

# 2. Show what's currently in env
env_token = os.environ.get("HF_TOKEN")
if env_token is None:
    print("❌ HF_TOKEN is not set in the environment at all")
elif env_token.startswith("hf_xxx"):
    print(f"❌ HF_TOKEN is the PLACEHOLDER — you copy-pasted without replacing")
    print(f"   value = {env_token}")
elif len(env_token) < 20:
    print(f"❌ HF_TOKEN looks too short: {env_token}")
elif not env_token.startswith("hf_"):
    print(f"❌ HF_TOKEN doesn't start with 'hf_': {env_token[:10]}...")
else:
    print(f"✅ HF_TOKEN looks like a real token: {env_token[:6]}...{env_token[-4:]} "
          f"(length {len(env_token)})")

# 3. Check Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient().get_secret("HF_TOKEN")
    if sec:
        print(f"⚠️  Kaggle Secret 'HF_TOKEN' exists: {sec[:6]}...{sec[-4:]}")
        print(f"   sync.py prefers this over env — it will be used even if env is correct")
    else:
        print(f"📝 No Kaggle Secret 'HF_TOKEN' (only env var will be used)")
except Exception as e:
    print(f"📝 Kaggle Secrets not available: {e}")

❌ HF_TOKEN looks too short: hf_PfJ...epWB
⚠️  Kaggle Secret 'HF_TOKEN' exists: hf_RlB...hpto
   sync.py prefers this over env — it will be used even if env is correct


In [20]:
import os
from huggingface_hub import HfApi

token = os.environ["HF_TOKEN"]
print(f"testing raw token: {token[:6]}...{token[-4:]} (len {len(token)})")

try:
    who = HfApi(token=token).whoami()
    print(f"✅ Token is VALID — logged in as: {who['name']}")
    print(f"   → the problem is sync.py caching the old token")
except Exception as e:
    print(f"❌ Token is INVALID — {e}")
    print(f"   → regenerate at https://huggingface.co/settings/tokens")

testing raw token: hf_PfJ...epWB (len 13)
❌ Token is INVALID — Invalid user token. The token from HF_TOKEN environment variable is invalid. Note that HF_TOKEN takes precedence over `hf auth login`.
   → regenerate at https://huggingface.co/settings/tokens


In [26]:
import os, sys

REAL_TOKEN = None
source = None

# 1. Try current env (in case it's still the correct one from earlier)
candidate = os.environ.get("HF_TOKEN", "")
if candidate.startswith("hf_") and len(candidate) == 37 and "REPLACE" not in candidate.upper() and "PASTE" not in candidate.upper():
    REAL_TOKEN = candidate
    source = "os.environ"

# 2. Try Kaggle Secrets
if REAL_TOKEN is None:
    try:
        from kaggle_secrets import UserSecretsClient
        candidate = UserSecretsClient().get_secret("HF_TOKEN")
        if candidate and candidate.startswith("hf_") and len(candidate) == 37:
            REAL_TOKEN = candidate
            source = "Kaggle Secrets"
    except Exception:
        pass

# 3. Try HF CLI cache (~/.cache/huggingface/token)
if REAL_TOKEN is None:
    from pathlib import Path
    for path in [
        Path.home() / ".cache" / "huggingface" / "token",
        Path.home() / ".huggingface" / "token",
    ]:
        if path.exists():
            candidate = path.read_text().strip()
            if candidate.startswith("hf_") and len(candidate) == 37:
                REAL_TOKEN = candidate
                source = f"cache ({path})"
                break

if REAL_TOKEN is None:
    print("❌ No valid token found anywhere.")
    print("   → Use Option A (getpass) instead, or")
    print("   → Regenerate at https://huggingface.co/settings/tokens")
    raise SystemExit(1)

print(f"✅ Found valid token from: {source}")
print(f"   {REAL_TOKEN[:6]}...{REAL_TOKEN[-4:]} ({len(REAL_TOKEN)} chars)")

# Verify it actually works
from huggingface_hub import HfApi
who = HfApi(token=REAL_TOKEN).whoami()
print(f"✅ Token authenticated as: {who['name']}")

os.environ["HF_TOKEN"] = REAL_TOKEN
os.environ["HF_USER"]  = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync
print(f"✅ sync reloaded, token = {sync.TOKEN[:6]}...{sync.TOKEN[-4:]}")

✅ Found valid token from: Kaggle Secrets
   hf_RlB...hpto (37 chars)
✅ Token authenticated as: OmarCodePlay
✅ sync reloaded, token = hf_RlB...hpto


In [21]:
import os
p = sync.pull_ckpt("best_base3.pt", "/kaggle/working/ckpt")
if p:
    size_mb = os.path.getsize(p) / 1e6
    print(f"✅ best_base3.pt at {p} ({size_mb:.1f} MB)")
else:
    print(f"❌ best_base3.pt not found on HF")

best_base3.pt: reconstructing file:   0%|          |  0.00B / 46.4MB            

best_base3.pt: downloading bytes:           |  0.00B            

✅ best_base3.pt at /kaggle/working/ckpt/best_base3.pt (46.4 MB)


In [29]:
import os, sys
os.environ["HF_TOKEN"] = "hf_RlB_REPLACE_WITH_FULL_TOKEN"   # ← your real token (from Kaggle Secrets)
os.environ["HF_USER"]  = "OmarCodePlay"

# OR fetch from Kaggle Secrets again (safer — no paste)
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import hf_hub_download

for i in ["0000", "0001", "0002"]:
    try:
        p = hf_hub_download(
            repo_id="Lichess/chess-puzzles",
            filename=f"default/train/{i}.parquet",
            repo_type="dataset",
            revision="refs/convert/parquet",
            token=os.environ["HF_TOKEN"],
            local_dir="/kaggle/working",
        )
        print(f"✅ {i}: {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
    except Exception as e:
        print(f"❌ {i}: {e}")

import glob
files = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
print(f"\n{len(files)} file(s) ready")

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B /  295MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

✅ 0000: /kaggle/working/default/train/0000.parquet  (294.9 MB)


default/train/0001.parquet: reconstructing file:   0%|          |  0.00B /  295MB            

default/train/0001.parquet: downloading bytes:           |  0.00B            

✅ 0001: /kaggle/working/default/train/0001.parquet  (294.8 MB)


default/train/0002.parquet: reconstructing file:   0%|          |  0.00B /  295MB            

default/train/0002.parquet: downloading bytes:           |  0.00B            

✅ 0002: /kaggle/working/default/train/0002.parquet  (294.8 MB)

3 file(s) ready


In [32]:
import os, sys, pathlib, glob, shutil

# ── 1. Copy src files from dataset mount (if present) ──
SRC_IN = "/kaggle/input/datasets/omarcodeplay/gm-dataset/src"
SRC_OUT = "/kaggle/working/src"
os.makedirs(SRC_OUT, exist_ok=True)

if os.path.isdir(SRC_IN):
    for f in pathlib.Path(SRC_IN).glob("*.py"):
        shutil.copy(f, pathlib.Path(SRC_OUT) / f.name)
    print(f"✅ copied {len(list(pathlib.Path(SRC_OUT).glob('*.py')))} files from {SRC_IN}")
else:
    print(f"⚠️  {SRC_IN} not mounted — will write build_puzzles.py manually")
    # If the dataset mount is gone, we still need encode.py and sync.py to import.
    # Check if they somehow survived
    for req in ["encode.py", "sync.py"]:
        if not (pathlib.Path(SRC_OUT) / req).exists():
            print(f"❌ missing required file: {req} — cannot proceed without it")
            print(f"   Re-attach the gm-dataset in the Kaggle sidebar")

# ── 2. Write build_puzzles.py (parquet-aware) ──
pathlib.Path("/kaggle/working/src/build_puzzles.py").write_text(r'''"""build_puzzles.py — Lichess puzzle parquet/CSV -> NPZ shards."""
import argparse, json, os
import chess, numpy as np, pandas as pd
from encode import pack_board, move_index, save_npz
import sync

SHARD = 500_000
THEME_VALUES = {
    "mateIn1":1.0,"mateIn2":1.0,"mateIn3":1.0,"mateIn4":1.0,"mateIn5":1.0,
    "smotheredMate":1.0,"anastasiaMate":1.0,"arabianMate":1.0,"backRankMate":1.0,
    "bodenMate":1.0,"cornerMate":1.0,"doubleBishopMate":1.0,"dovetailMate":1.0,
    "hookMate":1.0,"killBoxMate":1.0,"vukovicMate":1.0,
    "crushing":0.92,"sacrifice":0.90,"promotion":0.92,"doubleCheck":0.88,
    "fork":0.88,"pin":0.88,"skewer":0.88,"discoveredAttack":0.88,
    "xRayAttack":0.88,"attraction":0.88,"clearance":0.88,"interference":0.88,
    "intermezzo":0.88,"deflection":0.88,"overloading":0.88,"hangingPiece":0.88,
    "capturingDefender":0.88,"trappedPiece":0.88,"attackingF2F7":0.88,
    "enPassant":0.88,"zugzwang":0.82,"quietMove":0.82,
    "advantage":0.72,"attacking":0.72,"kingsideAttack":0.72,
    "queensideAttack":0.72,"exposedKing":0.72,
    "opening":0.65,"middlegame":0.65,"endgame":0.65,"rookEndgame":0.65,
    "pawnEndgame":0.65,"bishopEndgame":0.65,"knightEndgame":0.65,
    "queenEndgame":0.65,"queenRookEndgame":0.65,
    "defensiveMove":0.30,"counterCheck":0.30,"equality":0.00,
    "long":0.55,"short":0.55,"veryLong":0.55,"oneMove":0.55,
    "master":0.75,"masterVsMaster":0.75,"superGM":0.85,
    "advancedPawn":0.72,"collinearMove":0.55,"constellation":0.55,
}

def _to_theme_tokens(themes):
    if themes is None: return []
    if isinstance(themes, str): return themes.split()
    try: return [str(t) for t in themes]
    except TypeError: return str(themes).split()

def theme_to_value(themes):
    tokens = _to_theme_tokens(themes)
    if not tokens: return 0.90
    vals = [THEME_VALUES[t] for t in tokens if t in THEME_VALUES]
    return max(vals) if vals else 0.90

def _normalize_columns(df):
    rename = {
        "puzzleid":"PuzzleId","puzzle_id":"PuzzleId","fen":"FEN","moves":"Moves",
        "rating":"Rating","ratingdeviation":"RatingDeviation",
        "rating_deviation":"RatingDeviation","popularity":"Popularity",
        "nbplays":"NbPlays","nb_plays":"NbPlays","themes":"Themes",
        "gameurl":"GameUrl","game_url":"GameUrl",
        "openingtags":"OpeningTags","opening_tags":"OpeningTags",
        "dailydate":"DailyDate","daily_date":"DailyDate",
    }
    return df.rename(columns={c: rename.get(c.lower(), c) for c in df.columns})

def _load(files):
    parts = []
    for f in files:
        print(f"  loading {f} ...")
        if f.endswith(".parquet"):
            parts.append(pd.read_parquet(f))
        else:
            parts.append(pd.read_csv(f, low_memory=False))
    return _normalize_columns(pd.concat(parts, ignore_index=True))

def mate_moves(board):
    out = []
    for m in board.legal_moves:
        board.push(m); is_mate = board.is_checkmate(); board.pop()
        if is_mate: out.append(m)
    return out

def parse_puzzle(row, value_mode="themes"):
    fen = str(row.get("FEN","") or "")
    moves_str = str(row.get("Moves","") or "")
    themes = row.get("Themes")
    if not fen or not moves_str: return []
    try: board = chess.Board(fen)
    except Exception: return []
    moves = moves_str.split()
    if len(moves) < 2: return []
    try:
        setup = chess.Move.from_uci(moves[0])
        if setup not in board.legal_moves: return []
        board.push(setup)
    except Exception: return []
    v = theme_to_value(themes) if value_mode=="themes" else 0.95
    theme_set = set(_to_theme_tokens(themes))
    is_mate_in_one = ("mateIn1" in theme_set) or (len(moves)==2)
    out = []
    for i in range(1, len(moves), 2):
        try: sol = chess.Move.from_uci(moves[i])
        except Exception: break
        if sol not in board.legal_moves: break
        bb, meta = pack_board(board)
        black = board.turn == chess.BLACK
        if is_mate_in_one and i == 1:
            alternates = mate_moves(board)
            if len(alternates) > 1:
                alternates = alternates[:4]
                idxs = [move_index(m, black) for m in alternates]
                w = 1.0 / len(idxs)
                mvk = idxs + [-1]*(4-len(idxs))
                wk = [w]*len(idxs) + [0.0]*(4-len(idxs))
            else:
                mvk = [move_index(sol, black), -1, -1, -1]
                wk = [1.0, 0.0, 0.0, 0.0]
        else:
            mvk = [move_index(sol, black), -1, -1, -1]
            wk = [1.0, 0.0, 0.0, 0.0]
        out.append((bb, meta, np.array(mvk,dtype=np.int16),
                    np.array(wk,dtype=np.float16), float(v)))
        board.push(sol)
        if i+1 < len(moves):
            try:
                reply = chess.Move.from_uci(moves[i+1])
                if reply not in board.legal_moves: break
                board.push(reply)
            except Exception: break
        else: break
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--puzzles", nargs="+", required=True)
    ap.add_argument("--outdir", default="/kaggle/working/shards")
    ap.add_argument("--min-rating", type=int, default=1500)
    ap.add_argument("--max-rating", type=int, default=2500)
    ap.add_argument("--min-popularity", type=int, default=80)
    ap.add_argument("--min-plays", type=int, default=200)
    ap.add_argument("--max-rating-dev", type=int, default=100)
    ap.add_argument("--first-move-only", action="store_true")
    ap.add_argument("--value-mode", default="themes", choices=["themes","all_win"])
    ap.add_argument("--shard-prefix", default="puzz")
    ap.add_argument("--shard-size", type=int, default=SHARD)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--push", action="store_true")
    ap.add_argument("--update-index", action="store_true")
    args = ap.parse_args()

    print(f"Loading {len(args.puzzles)} file(s) ...")
    df = _load(args.puzzles)
    print(f"Loaded            {len(df):>10,} puzzles")
    print(f"Columns           {list(df.columns)}")

    df = df[(df["Rating"]>=args.min_rating) & (df["Rating"]<=args.max_rating)]
    print(f"After rating      {len(df):>10,}")
    if "RatingDeviation" in df.columns and args.max_rating_dev:
        df = df[df["RatingDeviation"]<=args.max_rating_dev]
        print(f"After max RD      {len(df):>10,}")
    if "Popularity" in df.columns:
        df = df[df["Popularity"].fillna(-100)>=args.min_popularity]
        print(f"After popularity  {len(df):>10,}")
    if "NbPlays" in df.columns:
        df = df[df["NbPlays"].fillna(0)>=args.min_plays]
        print(f"After plays       {len(df):>10,}")
    if args.limit:
        df = df.head(args.limit)
        print(f"After limit       {len(df):>10,}")

    if args.push: sync.init()
    all_rows = []; bad = multi = 0
    for r in df.to_dict("records"):
        rs = parse_puzzle(r, value_mode=args.value_mode)
        if args.first_move_only and rs: rs = rs[:1]
        if rs:
            if any(int(row[2][1])!=-1 for row in rs): multi += 1
            all_rows.extend(rs)
        else: bad += 1
    print(f"Samples           {len(all_rows):>10,}  "
          f"({bad:,} bad, {multi:,} with alternate mate solutions)")

    os.makedirs(args.outdir, exist_ok=True)
    shard_names = []
    for i in range(0, len(all_rows), args.shard_size):
        chunk = all_rows[i:i+args.shard_size]
        fn = f"{args.shard_prefix}_{i//args.shard_size:04d}.npz"
        local = os.path.join(args.outdir, fn)
        save_npz(chunk, local)
        print(f"  saved {fn}  ({len(chunk):,} rows)")
        if args.push:
            sync.push_data(local, fn)
            shard_names.append(fn)

    if args.update_index and args.push and shard_names:
        idx_path = sync.pull_data("index.json", "/kaggle/working/shards")
        with open(idx_path) as f: idx = json.load(f)
        existing = set(idx["train"])
        new = [n for n in shard_names if n not in existing]
        idx["train"].extend(new)
        with open("/tmp/index.json","w") as f: json.dump(idx, f)
        sync.push_data("/tmp/index.json", "index.json")
        print(f"index.json updated: +{len(new)} puzzle shards "
              f"(total train shards: {len(idx['train'])})")
    print("Done.")

if __name__ == "__main__":
    main()
''')
print("✅ build_puzzles.py written")

# ── 3. Re-patch sync.py for env precedence ──
sp = pathlib.Path("/kaggle/working/src/sync.py")
ssrc = sp.read_text()
old = '''try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN")'''
new = '''TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
if not TOKEN:
    raise RuntimeError("HF_TOKEN not found in env or Kaggle Secrets")'''
if old in ssrc:
    sp.write_text(ssrc.replace(old, new))
    print("✅ sync.py patched")
else:
    print("ℹ️  sync.py already patched")

# ── 4. Check parquet files ──
PUZZLE_FILES = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
if len(PUZZLE_FILES) == 3:
    print(f"\n✅ 3 parquet files present:")
    for p in PUZZLE_FILES:
        print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
else:
    print(f"\n⚠️  found {len(PUZZLE_FILES)} parquet files — re-downloading...")
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    from huggingface_hub import hf_hub_download
    for i in ["0000", "0001", "0002"]:
        try:
            p = hf_hub_download(
                repo_id="Lichess/chess-puzzles",
                filename=f"default/train/{i}.parquet",
                repo_type="dataset", revision="refs/convert/parquet",
                token=tok, local_dir="/kaggle/working")
            print(f"  ✅ {i}: {os.path.getsize(p)/1e6:.1f} MB")
        except Exception as e:
            print(f"  ❌ {i}: {e}")
    PUZZLE_FILES = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
    print(f"\n{len(PUZZLE_FILES)} file(s) ready")

✅ copied 11 files from /kaggle/input/datasets/omarcodeplay/gm-dataset/src
✅ build_puzzles.py written
✅ sync.py patched

✅ 3 parquet files present:
  /kaggle/working/default/train/0000.parquet  (294.9 MB)
  /kaggle/working/default/train/0001.parquet  (294.8 MB)
  /kaggle/working/default/train/0002.parquet  (294.8 MB)


In [33]:
import os, sys, subprocess, glob
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"]  = "OmarCodePlay"

env = os.environ.copy()
PUZZLE_FILES = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
assert len(PUZZLE_FILES) == 3, f"need 3 files, got {len(PUZZLE_FILES)}"

cmd = [
    sys.executable, "src/build_puzzles.py",
    "--puzzles", *PUZZLE_FILES,
    "--min-rating", "1500", "--max-rating", "2500",
    "--min-popularity", "80", "--min-plays", "200", "--max-rating-dev", "100",
    "--first-move-only", "--push", "--update-index",
]
print("Running build (~4-6 min)...\n")
r = subprocess.run(cmd, cwd="/kaggle/working", env=env,
                   capture_output=True, text=True, timeout=1800)
print(r.stdout)
if r.returncode != 0:
    print("STDERR (tail):", r.stderr[-1200:])
print("exit:", r.returncode)

Running build (~4-6 min)...

Loading 3 file(s) ...
  loading /kaggle/working/default/train/0000.parquet ...
  loading /kaggle/working/default/train/0001.parquet ...
  loading /kaggle/working/default/train/0002.parquet ...
Loaded             6,157,341 puzzles
Columns           ['PuzzleId', 'GameId', 'FEN', 'Moves', 'Rating', 'RatingDeviation', 'Popularity', 'NbPlays', 'Themes', 'OpeningTags']
After rating       2,532,866
After max RD       2,330,991
After popularity   2,050,316
After plays        1,583,968
Samples            1,583,968  (0 bad, 694 with alternate mate solutions)
  saved puzz_0000.npz  (500,000 rows)
  saved puzz_0001.npz  (500,000 rows)
  saved puzz_0002.npz  (500,000 rows)
  saved puzz_0003.npz  (83,968 rows)
index.json updated: +4 puzzle shards (total train shards: 6)
Done.

exit: 0


In [34]:
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"]  = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync

idx = json.load(open(sync.pull_data("index.json", "/kaggle/working/shards")))
games   = [s for s in idx["train"] if s.startswith("train_")]
puzzles = [s for s in idx["train"] if s.startswith("puzz_")]

g_est = len(games) * 450_000
p_est = len(puzzles) * 400_000
total = g_est + p_est

print(f"Train shards: {len(idx['train'])}")
for s in idx["train"]:
    print(f"  {s}")
print(f"\nGames:   {len(games)} shard(s)  (~{g_est:,} samples)")
print(f"Puzzles: {len(puzzles)} shard(s)  (~{p_est:,} samples)")
print(f"Total:   ~{total:,}")
print(f"Mix:     {100*g_est/total:.0f}% games / {100*p_est/total:.0f}% puzzles")

import torch
print(f"\nGPUs: {torch.cuda.device_count()}")

print(f"\nmax_steps options (12 epochs @ batch 2048):")
for n in [2, 3, 4]:
    t = g_est + n * 400_000
    ms = int(12 * t / 2048)
    print(f"  {n} puzzle shards:  total ~{t:,}  →  max_steps={ms:,}  "
          f"({100*g_est/t:.0f}% games)")

index.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Train shards: 6
  train_0000.npz
  train_0001.npz
  puzz_0000.npz
  puzz_0001.npz
  puzz_0002.npz
  puzz_0003.npz

Games:   2 shard(s)  (~900,000 samples)
Puzzles: 4 shard(s)  (~1,600,000 samples)
Total:   ~2,500,000
Mix:     36% games / 64% puzzles

GPUs: 2

max_steps options (12 epochs @ batch 2048):
  2 puzzle shards:  total ~1,700,000  →  max_steps=9,960  (53% games)
  3 puzzle shards:  total ~2,100,000  →  max_steps=12,304  (43% games)
  4 puzzle shards:  total ~2,500,000  →  max_steps=14,648  (36% games)


In [35]:
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"]  = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync

idx = json.load(open(sync.pull_data("index.json", "/kaggle/working/shards")))
games   = [s for s in idx["train"] if s.startswith("train_")]
puzzles = [s for s in idx["train"] if s.startswith("puzz_")][:2]   # keep 2 of 4

idx["train"] = games + puzzles
with open("/tmp/index.json", "w") as f:
    json.dump(idx, f)
sync.push_data("/tmp/index.json", "index.json")

print(f"✅ index.json trimmed")
print(f"   Games:   {len(games)} shard(s)  → {games}")
print(f"   Puzzles: {len(puzzles)} shard(s) → {puzzles}")
print(f"   Kept on HF (not in index): puzz_0002.npz, puzz_0003.npz")
print(f"   Use max_steps=10000, decay_start=7500, decay_steps=2500")

✅ index.json trimmed
   Games:   2 shard(s)  → ['train_0000.npz', 'train_0001.npz']
   Puzzles: 2 shard(s) → ['puzz_0000.npz', 'puzz_0001.npz']
   Kept on HF (not in index): puzz_0002.npz, puzz_0003.npz
   Use max_steps=10000, decay_start=7500, decay_steps=2500


In [36]:
import pathlib

p = pathlib.Path("/kaggle/working/src/train.py")
src = p.read_text()

src = src.replace("map_location=local_rank", 'map_location=f"cuda:{local_rank}"')
src = src.replace('run="base2",', 'run="base5",')
src = src.replace('init_from=None,', 'init_from="best_base3.pt",')
src = src.replace('lr=2e-3,', 'lr=3e-4,')
src = src.replace('wd=1e-4,', 'wd=5e-4,')
src = src.replace('warmup=1500,', 'warmup=200,')
src = src.replace('decay_start=None,', 'decay_start=7500,')
src = src.replace('decay_steps=100_000,', 'decay_steps=2500,')
src = src.replace('max_steps=None,', 'max_steps=10000,')
src = src.replace('smooth=0.05,', 'smooth=0.10,')
src = src.replace('eval_every=5000,', 'eval_every=500,')

if "patience" not in src:
    src = src.replace('max_steps=10000,',
                      'max_steps=10000,\n    patience=3,\n    valmse_tolerance=0.001,')

src = src.replace(
    '        if "best" not in st:\n            st["best"] = 1e9',
    '        if "best" not in st:\n            st["best"] = 1e9\n'
    '        if "valmse_last" not in st:\n            st["valmse_last"] = None\n'
    '        if "valmse_worse" not in st:\n            st["valmse_worse"] = 0'
)

old_eval = (
    '                    pl, vl, t1 = evaluate(ema, val)\n'
    '                    say(\n'
    '                        f"EVAL step {st[\'step\']} pol {pl:.3f} "\n'
    '                        f"valmse {vl:.4f} top1 {t1:.3f}"\n'
    '                    )\n'
    '                    score = pl + C["v_w"] * vl'
)
new_eval = (
    '                    pl, vl, t1 = evaluate(ema, val)\n'
    '                    score = pl + C["v_w"] * vl\n'
    '                    prev_v = st.get("valmse_last")\n'
    '                    if prev_v is not None and vl > prev_v + C.get("valmse_tolerance", 0.001):\n'
    '                        st["valmse_worse"] = st.get("valmse_worse", 0) + 1\n'
    '                    else:\n'
    '                        st["valmse_worse"] = 0\n'
    '                    st["valmse_last"] = vl\n'
    '                    say(\n'
    '                        f"EVAL step {st[\'step\']} pol {pl:.3f} "\n'
    '                        f"valmse {vl:.4f} top1 {t1:.3f} "\n'
    '                        f"worse {st[\'valmse_worse\']}/{C.get(\'patience\', 3)}"\n'
    '                    )'
)
if old_eval in src:
    src = src.replace(old_eval, new_eval)
else:
    print("WARN: eval block not found")

old_stop = (
    '                        or (C["max_steps"] and st["step"] >= C["max_steps"])\n'
    '                    )\n'
    '                ]'
)
new_stop = (
    '                        or (C["max_steps"] and st["step"] >= C["max_steps"])\n'
    '                        or (C.get("patience", 0) > 0\n'
    '                            and st.get("valmse_worse", 0) >= C["patience"])\n'
    '                    )\n'
    '                ]'
)
if old_stop in src:
    src = src.replace(old_stop, new_stop)
else:
    print("WARN: stop block not found")

p.write_text(src)
print("✅ train.py patched")

!grep -nE 'run=|init_from=|lr=|max_steps=|patience=|decay_start=' /kaggle/working/src/train.py | head -10

✅ train.py patched
26:    run="base5",          # keep this to resume from last_base2.pt
27:    init_from="best_base3.pt",
31:    lr=3e-4,
34:    decay_start=7500,
44:    max_steps=10000,
45:    patience=3,
151:    opt = torch.optim.AdamW(model.parameters(), lr=C["lr"], weight_decay=C["wd"])


In [37]:
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"]  = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]

!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee /kaggle/working/train_base5.log

W1111 17:16:24.011000 288 torch/distributed/run.py:851] 
W1111 17:16:24.011000 288 torch/distributed/run.py:851] *****************************************
W1111 17:16:24.011000 288 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1111 17:16:24.011000 288 torch/distributed/run.py:851] *****************************************
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1111 17:16:27.657495643 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
CONFIG {'run': 'base

In [38]:
import os, sys, pathlib, shutil, glob

# ── Copy src ──
SRC_IN = "/kaggle/input/datasets/omarcodeplay/gm-dataset/src"
SRC_OUT = "/kaggle/working/src"
os.makedirs(SRC_OUT, exist_ok=True)
if os.path.isdir(SRC_IN):
    for f in pathlib.Path(SRC_IN).glob("*.py"):
        shutil.copy(f, pathlib.Path(SRC_OUT) / f.name)
    print(f"✅ copied {len(list(pathlib.Path(SRC_OUT).glob('*.py')))} src files")
else:
    print(f"❌ {SRC_IN} not mounted — re-attach gm-dataset in sidebar")

# ── Write build_puzzles.py ──
pathlib.Path("/kaggle/working/src/build_puzzles.py").write_text(r'''"""build_puzzles.py — Lichess puzzle parquet/CSV -> NPZ shards."""
import argparse, json, os
import chess, numpy as np, pandas as pd
from encode import pack_board, move_index, save_npz
import sync

SHARD = 500_000
THEME_VALUES = {
    "mateIn1":1.0,"mateIn2":1.0,"mateIn3":1.0,"mateIn4":1.0,"mateIn5":1.0,
    "smotheredMate":1.0,"anastasiaMate":1.0,"arabianMate":1.0,"backRankMate":1.0,
    "bodenMate":1.0,"cornerMate":1.0,"doubleBishopMate":1.0,"dovetailMate":1.0,
    "hookMate":1.0,"killBoxMate":1.0,"vukovicMate":1.0,
    "crushing":0.92,"sacrifice":0.90,"promotion":0.92,"doubleCheck":0.88,
    "fork":0.88,"pin":0.88,"skewer":0.88,"discoveredAttack":0.88,
    "xRayAttack":0.88,"attraction":0.88,"clearance":0.88,"interference":0.88,
    "intermezzo":0.88,"deflection":0.88,"overloading":0.88,"hangingPiece":0.88,
    "capturingDefender":0.88,"trappedPiece":0.88,"attackingF2F7":0.88,
    "enPassant":0.88,"zugzwang":0.82,"quietMove":0.82,
    "advantage":0.72,"attacking":0.72,"kingsideAttack":0.72,
    "queensideAttack":0.72,"exposedKing":0.72,
    "opening":0.65,"middlegame":0.65,"endgame":0.65,"rookEndgame":0.65,
    "pawnEndgame":0.65,"bishopEndgame":0.65,"knightEndgame":0.65,
    "queenEndgame":0.65,"queenRookEndgame":0.65,
    "defensiveMove":0.30,"counterCheck":0.30,"equality":0.00,
    "long":0.55,"short":0.55,"veryLong":0.55,"oneMove":0.55,
    "master":0.75,"masterVsMaster":0.75,"superGM":0.85,
    "advancedPawn":0.72,"collinearMove":0.55,"constellation":0.55,
}

def _to_theme_tokens(themes):
    if themes is None: return []
    if isinstance(themes, str): return themes.split()
    try: return [str(t) for t in themes]
    except TypeError: return str(themes).split()

def theme_to_value(themes):
    tokens = _to_theme_tokens(themes)
    if not tokens: return 0.90
    vals = [THEME_VALUES[t] for t in tokens if t in THEME_VALUES]
    return max(vals) if vals else 0.90

def _normalize_columns(df):
    rename = {
        "puzzleid":"PuzzleId","puzzle_id":"PuzzleId","fen":"FEN","moves":"Moves",
        "rating":"Rating","ratingdeviation":"RatingDeviation",
        "rating_deviation":"RatingDeviation","popularity":"Popularity",
        "nbplays":"NbPlays","nb_plays":"NbPlays","themes":"Themes",
        "gameurl":"GameUrl","game_url":"GameUrl",
        "openingtags":"OpeningTags","opening_tags":"OpeningTags",
        "dailydate":"DailyDate","daily_date":"DailyDate",
    }
    return df.rename(columns={c: rename.get(c.lower(), c) for c in df.columns})

def _load(files):
    parts = []
    for f in files:
        print(f"  loading {f} ...")
        if f.endswith(".parquet"):
            parts.append(pd.read_parquet(f))
        else:
            parts.append(pd.read_csv(f, low_memory=False))
    return _normalize_columns(pd.concat(parts, ignore_index=True))

def mate_moves(board):
    out = []
    for m in board.legal_moves:
        board.push(m); is_mate = board.is_checkmate(); board.pop()
        if is_mate: out.append(m)
    return out

def parse_puzzle(row, value_mode="themes"):
    fen = str(row.get("FEN","") or "")
    moves_str = str(row.get("Moves","") or "")
    themes = row.get("Themes")
    if not fen or not moves_str: return []
    try: board = chess.Board(fen)
    except Exception: return []
    moves = moves_str.split()
    if len(moves) < 2: return []
    try:
        setup = chess.Move.from_uci(moves[0])
        if setup not in board.legal_moves: return []
        board.push(setup)
    except Exception: return []
    v = theme_to_value(themes) if value_mode=="themes" else 0.95
    theme_set = set(_to_theme_tokens(themes))
    is_mate_in_one = ("mateIn1" in theme_set) or (len(moves)==2)
    out = []
    for i in range(1, len(moves), 2):
        try: sol = chess.Move.from_uci(moves[i])
        except Exception: break
        if sol not in board.legal_moves: break
        bb, meta = pack_board(board)
        black = board.turn == chess.BLACK
        if is_mate_in_one and i == 1:
            alternates = mate_moves(board)
            if len(alternates) > 1:
                alternates = alternates[:4]
                idxs = [move_index(m, black) for m in alternates]
                w = 1.0 / len(idxs)
                mvk = idxs + [-1]*(4-len(idxs))
                wk = [w]*len(idxs) + [0.0]*(4-len(idxs))
            else:
                mvk = [move_index(sol, black), -1, -1, -1]
                wk = [1.0, 0.0, 0.0, 0.0]
        else:
            mvk = [move_index(sol, black), -1, -1, -1]
            wk = [1.0, 0.0, 0.0, 0.0]
        out.append((bb, meta, np.array(mvk,dtype=np.int16),
                    np.array(wk,dtype=np.float16), float(v)))
        board.push(sol)
        if i+1 < len(moves):
            try:
                reply = chess.Move.from_uci(moves[i+1])
                if reply not in board.legal_moves: break
                board.push(reply)
            except Exception: break
        else: break
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--puzzles", nargs="+", required=True)
    ap.add_argument("--outdir", default="/kaggle/working/shards")
    ap.add_argument("--min-rating", type=int, default=1500)
    ap.add_argument("--max-rating", type=int, default=2500)
    ap.add_argument("--min-popularity", type=int, default=80)
    ap.add_argument("--min-plays", type=int, default=200)
    ap.add_argument("--max-rating-dev", type=int, default=100)
    ap.add_argument("--first-move-only", action="store_true")
    ap.add_argument("--value-mode", default="themes", choices=["themes","all_win"])
    ap.add_argument("--shard-prefix", default="puzz")
    ap.add_argument("--shard-size", type=int, default=SHARD)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--push", action="store_true")
    args = ap.parse_args()

    print(f"Loading {len(args.puzzles)} file(s) ...")
    df = _load(args.puzzles)
    print(f"Loaded            {len(df):>10,} puzzles")
    print(f"Columns           {list(df.columns)}")
    df = df[(df["Rating"]>=args.min_rating) & (df["Rating"]<=args.max_rating)]
    print(f"After rating      {len(df):>10,}")
    if "RatingDeviation" in df.columns and args.max_rating_dev:
        df = df[df["RatingDeviation"]<=args.max_rating_dev]
        print(f"After max RD      {len(df):>10,}")
    if "Popularity" in df.columns:
        df = df[df["Popularity"].fillna(-100)>=args.min_popularity]
        print(f"After popularity  {len(df):>10,}")
    if "NbPlays" in df.columns:
        df = df[df["NbPlays"].fillna(0)>=args.min_plays]
        print(f"After plays       {len(df):>10,}")
    if args.limit:
        df = df.head(args.limit)
        print(f"After limit       {len(df):>10,}")

    if args.push: sync.init()
    all_rows = []; bad = multi = 0
    for r in df.to_dict("records"):
        rs = parse_puzzle(r, value_mode=args.value_mode)
        if args.first_move_only and rs: rs = rs[:1]
        if rs:
            if any(int(row[2][1])!=-1 for row in rs): multi += 1
            all_rows.extend(rs)
        else: bad += 1
    print(f"Samples           {len(all_rows):>10,}  "
          f"({bad:,} bad, {multi:,} with alternate mate solutions)")

    os.makedirs(args.outdir, exist_ok=True)
    shard_names = []
    for i in range(0, len(all_rows), args.shard_size):
        chunk = all_rows[i:i+args.shard_size]
        fn = f"{args.shard_prefix}_{i//args.shard_size:04d}.npz"
        local = os.path.join(args.outdir, fn)
        save_npz(chunk, local)
        print(f"  saved {fn}  ({len(chunk):,} rows)")
        if args.push:
            sync.push_data(local, fn)
            shard_names.append(fn)
    print(f"Pushed {len(shard_names)} shards to HF")
    print("Done.")

if __name__ == "__main__":
    main()
''')
print("✅ build_puzzles.py written")

# ── Patch sync.py for env precedence ──
sp = pathlib.Path("/kaggle/working/src/sync.py")
ssrc = sp.read_text()
old = '''try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN")'''
new = '''TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
if not TOKEN:
    raise RuntimeError("HF_TOKEN not found")'''
if old in ssrc:
    sp.write_text(ssrc.replace(old, new))
    print("✅ sync.py patched")
else:
    print("ℹ️  sync.py already patched")

✅ copied 12 src files
✅ build_puzzles.py written
✅ sync.py patched


In [ ]:
import os, sys, subprocess, glob

env = os.environ.copy()
PUZZLE_FILES = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
assert len(PUZZLE_FILES) == 3

cmd = [
    sys.executable, "src/build_puzzles.py",
    "--puzzles", *PUZZLE_FILES,
    "--min-rating", "1500", "--max-rating", "2500",
    "--min-popularity", "80", "--min-plays", "200", "--max-rating-dev", "100",
    "--first-move-only",
    "--push",
]
print("Running build...\n")
r = subprocess.run(cmd, cwd="/kaggle/working", env=env,
                   capture_output=True, text=True, timeout=1800)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-1000:])
print("exit:", r.returncode)

Running build...



In [ ]:
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"]  = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync

idx = {
    "train": ["puzz_0000.npz", "puzz_0001.npz", "puzz_0002.npz"],  # 1.5M
    "val":   ["puzz_0003.npz"],                                      # 84k
    "test":  ["puzz_0003.npz"],                                      # reuse for test
}

with open("/tmp/index.json", "w") as f:
    json.dump(idx, f)
sync.push_data("/tmp/index.json", "index.json")

print("✅ puzzle-only index.json pushed")
print(f"   train: {idx['train']}  (~1.5M samples)")
print(f"   val:   {idx['val']}    (~84k samples)")
print(f"   test:  {idx['test']}   (same as val)")

In [ ]:
# Save current (games+puzzles) index as backup before overwriting
# (only needed the first time you run this)
import shutil, os
backup = "/kaggle/working/index_games_backup.json"
if not os.path.exists(backup):
    # Re-derive games-only index from what we know
    games_idx = {
        "train": ["train_0000.npz", "train_0001.npz"],
        "val":   ["val.npz"],
        "test":  ["test.npz"],
    }
    import json
    with open(backup, "w") as f:
        json.dump(games_idx, f)
    print(f"✅ games-only index saved to {backup}")
    print(f"   (revert later with: sync.push_data('{backup}', 'index.json'))")

In [ ]:
import pathlib
pathlib.Path("/kaggle/working/src/train.py").write_text(r'''"""DDP training — puzzle-only config (v_w=0.1 to protect value head)."""
import copy, json, math, os, time
import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from model import ChessNet
import sync

os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_SHM_DISABLE", "1")
torch.backends.cudnn.benchmark = True
KEYS = ("bb", "meta", "mvk", "wk", "val")

C = dict(
    run="puzz1",                 # new run name — no conflict with baseN
    init_from="engine.pt",
    filters=128, blocks=10,
    batch=2048,
    lr=5e-5,                     # gentle fine-tune (5e-5, not 3e-4)
    wd=1e-4,                     # smaller weight decay too
    warmup=500,                  # 2.5x base5's warmup
    decay_start=3500,            # cosine starts at 70%
    decay_steps=1500,
    max_steps=5000,
    v_w=0.1,                     # ⚠️ KEY: only 10% value weight (puzzles are all +)
    ema=0.999, smooth=0.05,
    shards_per_chunk=8,
    session_h=10.5, ckpt_min=20, push_min=60,
    eval_every=250,              # more frequent since run is short
)

WORK = os.environ.get("WORK", "/kaggle/working")
CK = f"{WORK}/ckpt"
DATA = f"{WORK}/shards"


def lr_at(step):
    lr = C["lr"] * min(1.0, (step + 1) / C["warmup"])
    if C["decay_start"] is not None and step >= C["decay_start"]:
        t = min(1.0, (step - C["decay_start"]) / C["decay_steps"])
        lr *= 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * t))
    return lr


def load_chunk(names, device):
    arr = {k: [] for k in KEYS}
    local = os.path.join(DATA, f"r{device}")
    os.makedirs(local, exist_ok=True)
    for n in names:
        p = os.path.join(local, n)
        if not os.path.exists(p):
            p = sync.pull_data(n, local)
        z = np.load(p)
        for k in KEYS:
            arr[k].append(z[k])
    return {k: torch.from_numpy(np.concatenate(v)).to(device) for k, v in arr.items()}


def pol_loss(lg, mvk, wk, smooth):
    lp = F.log_softmax(lg.float(), 1)
    w = wk.float()
    w = w / w.sum(1, keepdim=True).clamp_min(1e-6)
    nll = -(w * lp.gather(1, mvk.long().clamp_min(0))).sum(1).mean()
    return (1 - smooth) * nll + smooth * (-lp.mean(1)).mean()


@torch.no_grad()
def evaluate(m, d):
    m.eval()
    n = d["val"].shape[0]
    top1 = 0; pl = 0.0; vl = 0.0
    for i in range(0, n, 4096):
        sl = slice(i, i + 4096)
        with torch.autocast("cuda", dtype=torch.float16):
            lg, pv = m(d["bb"][sl], d["meta"][sl])
        mv = d["mvk"][sl, 0].long()
        pl += F.cross_entropy(lg.float(), mv, reduction="sum").item()
        top1 += (lg.argmax(1) == mv).sum().item()
        vl += F.mse_loss(pv.float(), d["val"][sl], reduction="sum").item()
    return pl / n, vl / n, top1 / n


@torch.no_grad()
def ema_update(ema, m, d):
    for pe, pm in zip(ema.parameters(), m.parameters()):
        pe.mul_(d).add_(pm.detach(), alpha=1 - d)
    for be, bm in zip(ema.buffers(), m.buffers()):
        be.copy_(bm)


def main():
    global C
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    local_rank = int(os.environ.get("LOCAL_RANK", rank))
    world = dist.get_world_size()
    torch.cuda.set_device(local_rank)
    torch.manual_seed(1234 + rank)
    np.random.seed(1234 + rank)
    is_main = rank == 0

    os.makedirs(CK, exist_ok=True)
    os.makedirs(DATA, exist_ok=True)
    if is_main: sync.init()
    dist.barrier()

    if is_main:
        idx = json.load(open(sync.pull_data("index.json", DATA)))
    else:
        idx = None
    box = [C, idx]
    dist.broadcast_object_list(box, src=0)
    C, idx = box[0], box[1]
    name = C["run"]
    if is_main: print("CONFIG", C, flush=True)

    raw = ChessNet(C["filters"], C["blocks"]).to(local_rank) \
        .to(memory_format=torch.channels_last)
    ema = copy.deepcopy(raw)
    model = nn.SyncBatchNorm.convert_sync_batchnorm(raw)
    opt = torch.optim.AdamW(model.parameters(), lr=C["lr"], weight_decay=C["wd"])
    scaler = torch.amp.GradScaler("cuda")
    st = dict(step=0, epoch=0, cursor=0, best=1e9)

    if is_main:
        p = sync.pull_ckpt(f"last_{name}.pt", CK)
    else:
        p = None
    pbox = [p]
    dist.broadcast_object_list(pbox, src=0)
    p = pbox[0]
    if p:
        ck = torch.load(p, map_location=f"cuda:{local_rank}", weights_only=False)
        model.load_state_dict(ck["model"])
        ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"])
        st = ck["st"]
        st.setdefault("best", 1e9)
        if is_main:
            print(f"RESUMED at step {st['step']} best {st['best']}", flush=True)
    elif C["init_from"]:
        ip = sync.pull_ckpt(C["init_from"], CK) if is_main else None
        ibox = [ip]
        dist.broadcast_object_list(ibox, src=0)
        ip = ibox[0]
        sd = torch.load(ip, map_location=f"cuda:{local_rank}", weights_only=False)["sd"]
        sd = {k.replace("module.", ""): v for k, v in sd.items()}
        model.load_state_dict(sd)
        ema.load_state_dict(sd)
        if is_main: print(f"INIT from {C['init_from']}", flush=True)

    ddp = DDP(model, device_ids=[local_rank])
    val = load_chunk(idx["val"], local_rank)

    def save(push):
        if not is_main: return
        tmp = f"{CK}/last.tmp"
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(),
                        opt=opt.state_dict(), scaler=scaler.state_dict(),
                        st=st), tmp)
        os.replace(tmp, f"{CK}/last_{name}.pt")
        torch.save(dict(filters=C["filters"], blocks=C["blocks"],
                        sd=ema.state_dict()), f"{CK}/ema_{name}.pt")
        if push:
            for f in (f"last_{name}.pt", f"ema_{name}.pt", f"best_{name}.pt"):
                if os.path.exists(f"{CK}/{f}"):
                    sync.push_ckpt(f"{CK}/{f}")

    t0 = last_ck = last_push = time.time()
    done = False
    seen = 0
    B = C["batch"] // world
    log = open(f"{CK}/train.log", "a") if is_main else None

    def say(msg):
        if is_main:
            print(msg, flush=True)
            log.write(msg + "\n")
            log.flush()

    while not done:
        order = np.random.default_rng(st["epoch"]).permutation(len(idx["train"]))
        while st["cursor"] < len(order) and not done:
            names = [idx["train"][i] for i in
                     order[st["cursor"]: st["cursor"] + C["shards_per_chunk"]]]
            data = load_chunk(names, local_rank)
            n = data["val"].shape[0]
            per = n // world
            lo = rank * per
            perm_local = torch.randperm(per, device=local_rank) + lo
            ddp.train()
            for i in range(0, per - B + 1, B):
                b = perm_local[i: i + B]
                bb, meta, mvk, wk, v = [data[k][b] for k in KEYS]
                for g in opt.param_groups:
                    g["lr"] = lr_at(st["step"])
                with torch.autocast("cuda", dtype=torch.float16):
                    lg, pv = ddp(bb, meta)
                lp = pol_loss(lg, mvk, wk, C["smooth"])
                lv = F.mse_loss(pv.float(), v)
                loss = lp + C["v_w"] * lv
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt)
                scaler.update()
                st["step"] += 1
                seen += B * world
                ema_update(ema, model, C["ema"])
                if st["step"] % 250 == 0:
                    say(f"step {st['step']} pol {lp.item():.3f} "
                        f"val {lv.item():.4f} lr {lr_at(st['step']):.2e} "
                        f"pos/s {seen / (time.time() - t0):.0f}")
                if st["step"] % C["eval_every"] == 0 and is_main:
                    pl, vl, t1 = evaluate(ema, val)
                    score = pl + C["v_w"] * vl
                    say(f"EVAL step {st['step']} pol {pl:.3f} "
                        f"valmse {vl:.4f} top1 {t1:.3f}")
                    if score < st["best"]:
                        st["best"] = score
                        torch.save(dict(filters=C["filters"], blocks=C["blocks"],
                                        sd=ema.state_dict()),
                                   f"{CK}/best_{name}.pt")
                now = time.time()
                if now - last_ck > C["ckpt_min"] * 60:
                    save(False); last_ck = now
                if now - last_push > C["push_min"] * 60:
                    save(True); last_push = now
                stop = [is_main and (
                    now - t0 > C["session_h"] * 3600
                    or (C["max_steps"] and st["step"] >= C["max_steps"])
                )]
                dist.broadcast_object_list(stop, src=0)
                if stop[0]:
                    done = True; break
            if not done:
                st["cursor"] += len(names)
            del data
            torch.cuda.empty_cache()
        if not done:
            st["epoch"] += 1
            st["cursor"] = 0

    save(True)
    say(f"SESSION END step {st['step']}")
    dist.destroy_process_group()


if __name__ == "__main__":
    main()
''')
print("✅ train.py written (puzzle-only config)")